In [1]:
# Python version
import sys
print(sys.version)

3.12.13 | packaged by Anaconda, Inc. | (main, Mar 19 2026, 20:16:45) [MSC v.1942 64 bit (AMD64)]


In [2]:
# Import key libraries and check versions
import pandas as pd
import numpy as np
import sklearn
import irapy

print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("sklearn:", sklearn.__version__)
print("irapy:", irapy.__version__)

pandas: 2.3.3
numpy: 2.2.5
sklearn: 1.8.0
irapy: 0.1.2


In [3]:
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.svm import SVR
from sklearn.linear_model import Ridge
import time
from sklearn.pipeline import Pipeline
from sklearn.datasets import fetch_california_housing
from sklearn.datasets import make_friedman1
from sklearn.datasets import make_regression
from irapy import single_ira
from sklearn.linear_model import LinearRegression
import warnings
warnings.filterwarnings("ignore")

In [4]:
# Runtime for the synthetic linear example
def synthetic_linear():
    X, y, coef = make_regression(
        n_samples=5000,
        n_features=8,
        noise = 50,
        bias = 100,
        n_informative=5,
        coef=True,
        random_state=42
    )

    X_ = pd.DataFrame(X,columns = [f"X{i}" for i in range(1,9,1)])
    y_ = pd.Series(y,name = 'y')

    model = LinearRegression().fit(X_, y_)

    runtime_list = []
    for i in range(1,31):
        start_time = time.time()
        single_ira(X_, model, num_interpol=100, num_background_samples=200, random_state=i)
        end_time = time.time()
        runtime_list.append(end_time-start_time)
    return pd.DataFrame({'Linear':runtime_list})

In [5]:
synthetic_linear_df = synthetic_linear()
synthetic_linear_df

,Linear
0,0.119325
1,0.067546
2,0.065771
3,0.060163
4,0.064900
5,0.059966
6,0.062512
7,0.047293
8,0.061116
9,0.063710


In [6]:
# Runtime for the synthetic nonlinear example
def synthetic_nonlinear():
    X, y = make_friedman1(
        n_samples=5000,
        n_features=8,
        noise=6,
        random_state=42
    )
    X_ = pd.DataFrame(X,columns = [f"X{i}" for i in range(1,9,1)])
    y_ = pd.Series(y,name = 'y')

    model = RandomForestRegressor(random_state=42).fit(X_, y_)

    runtime_list = []
    for i in range(1,31):
        start_time = time.time()
        single_ira(X_, model, num_interpol=100,  num_background_samples=200, random_state=i)
        end_time = time.time()
        runtime_list.append(end_time-start_time)
    return pd.DataFrame({'Nonlinear':runtime_list})

In [7]:
synthetic_nonlinear_df = synthetic_nonlinear()
synthetic_nonlinear_df

,Nonlinear
0,0.741891
1,0.733774
2,0.770223
3,0.800606
4,0.742492
5,0.829823
6,0.757362
7,0.792913
8,0.737338
9,0.811717


In [8]:
# Runtime for the California Housing example
def california_housing():
    data = fetch_california_housing(as_frame= True)
    X = data.data
    y = data.target

    feature_name_map = {
        "MedInc": "Median Income",
        "HouseAge": "House Age",
        "AveRooms": "Average Rooms",
        "AveBedrms": "Average Bedrooms",
        "Population": "Population",
        "AveOccup": "Average Occupancy",
        "Latitude": "Latitude",
        "Longitude": "Longitude"
    }

    X_ = X.rename(columns = feature_name_map)

    y_ = y.rename("Median House Value")
    
    models = {
    'GB': GradientBoostingRegressor(random_state=42),

    'MLP': Pipeline([
        ('scaler', StandardScaler()),
        ('MLP', MLPRegressor(random_state=42, max_iter=500)),
    ]),

    'RF': RandomForestRegressor(random_state=42),

    'RR': Pipeline([
        ('scaler', StandardScaler()),
        ('RR', Ridge()),
    ]),

    'SVR': Pipeline([
        ('scaler', StandardScaler()),
        ('SVR', SVR()),
    ])
    }


    final_result = {}
    for name, model in models.items():
        model = model.fit(X_, y_)
        print(name)
        runtime_list = []
        for i in range(1,31):
            start_time = time.time()
            single_ira(X_, model, num_interpol=100,  num_background_samples=200, random_state=i)
            end_time = time.time()
            runtime_list.append(end_time-start_time)
        
        final_result["CH_"+name]=runtime_list
    return pd.DataFrame(final_result)

In [9]:
california_housing_df = california_housing()
california_housing_df

GB
MLP
RF
RR
SVR


,CH_GB,CH_MLP,CH_RF,CH_RR,CH_SVR
0,0.185501,0.166911,0.873888,0.076798,159.750096
1,0.190706,0.203406,0.874056,0.079305,164.094860
2,0.185017,0.193104,0.859605,0.067569,166.764672
3,0.187760,0.171423,0.827828,0.080822,165.762035
4,0.185753,0.162907,0.844696,0.064255,169.135525
5,0.189494,0.144182,0.858462,0.068951,169.383820
6,0.170806,0.156217,0.844018,0.099629,165.437355
7,0.193840,0.155795,0.836753,0.087687,164.094631
8,0.166540,0.141336,0.836029,0.075354,181.751593
9,0.200113,0.157517,0.859500,0.065145,164.951585


In [10]:
# Runtime for the Concrete Compressive Strength example
def concrete_compressive_strength():
    
    # Download Concrete_Data.xls from:
    # https://archive.ics.uci.edu/dataset/165/concrete+compressive+strength
    df = pd.read_excel("Concrete_Data.xls")
    
    df.columns = [
        "Cement",
        "Blast Furnace Slag",
        "Fly Ash",
        "Water",
        "Superplasticizer",
        "Coarse Aggregate",
        "Fine Aggregate",
        "Age",
        "Compressive Strength"
    ]

    X = df.drop(columns=["Compressive Strength"])
    y = df["Compressive Strength"]
    
    models = {
    'GB': GradientBoostingRegressor(random_state=42),

    'MLP': Pipeline([
        ('scaler', StandardScaler()),
        ('MLP', MLPRegressor(random_state=42, max_iter=500)),
    ]),

    'RF': RandomForestRegressor(random_state=42),

    'RR': Pipeline([
        ('scaler', StandardScaler()),
        ('RR', Ridge()),
    ]),

    'SVR': Pipeline([
        ('scaler', StandardScaler()),
        ('SVR', SVR()),
    ])
    }


    final_result = {}
    for name, model in models.items():
        model = model.fit(X, y)
        print(name)
        runtime_list = []
        for i in range(1,31):
            start_time = time.time()
            single_ira(X, model, num_interpol=100, num_background_samples=200, random_state=i)
            end_time = time.time()
            runtime_list.append(end_time-start_time)
        
        final_result["CCS_"+name]=runtime_list
    return pd.DataFrame(final_result)

In [11]:
concrete_compressive_strength_df = concrete_compressive_strength()
concrete_compressive_strength_df

GB
MLP
RF
RR
SVR


,CCS_GB,CCS_MLP,CCS_RF,CCS_RR,CCS_SVR
0,0.188149,0.167441,0.531709,0.102299,11.049460
1,0.182528,0.172899,0.529984,0.121294,10.759974
2,0.174325,0.163419,0.579261,0.101967,10.703108
3,0.188380,0.155817,0.514996,0.126126,10.611916
4,0.188422,0.165270,0.560916,0.096176,10.567583
5,0.182689,0.150175,0.533313,0.094007,10.530535
6,0.184105,0.141101,0.530285,0.100453,10.541993
7,0.179762,0.149503,0.563960,0.170732,10.560561
8,0.187066,0.146256,0.624187,0.109621,10.579836
9,0.187004,0.148387,0.562384,0.114059,10.562323


In [12]:
runtime_df = pd.concat([synthetic_linear_df,
                         synthetic_nonlinear_df,
                         california_housing_df,
                         concrete_compressive_strength_df],axis=1)
runtime_df

,Linear,Nonlinear,CH_GB,CH_MLP,CH_RF,CH_RR,CH_SVR,CCS_GB,CCS_MLP,CCS_RF,CCS_RR,CCS_SVR
0,0.119325,0.741891,0.185501,0.166911,0.873888,0.076798,159.750096,0.188149,0.167441,0.531709,0.102299,11.049460
1,0.067546,0.733774,0.190706,0.203406,0.874056,0.079305,164.094860,0.182528,0.172899,0.529984,0.121294,10.759974
2,0.065771,0.770223,0.185017,0.193104,0.859605,0.067569,166.764672,0.174325,0.163419,0.579261,0.101967,10.703108
3,0.060163,0.800606,0.187760,0.171423,0.827828,0.080822,165.762035,0.188380,0.155817,0.514996,0.126126,10.611916
4,0.064900,0.742492,0.185753,0.162907,0.844696,0.064255,169.135525,0.188422,0.165270,0.560916,0.096176,10.567583
5,0.059966,0.829823,0.189494,0.144182,0.858462,0.068951,169.383820,0.182689,0.150175,0.533313,0.094007,10.530535
6,0.062512,0.757362,0.170806,0.156217,0.844018,0.099629,165.437355,0.184105,0.141101,0.530285,0.100453,10.541993
7,0.047293,0.792913,0.193840,0.155795,0.836753,0.087687,164.094631,0.179762,0.149503,0.563960,0.170732,10.560561
8,0.061116,0.737338,0.166540,0.141336,0.836029,0.075354,181.751593,0.187066,0.146256,0.624187,0.109621,10.579836
9,0.063710,0.811717,0.200113,0.157517,0.859500,0.065145,164.951585,0.187004,0.148387,0.562384,0.114059,10.562323


In [13]:
# Mean
print(runtime_df.mean())

Linear         0.064557
Nonlinear      0.751025
CH_GB          0.183311
CH_MLP         0.156103
CH_RF          0.848892
CH_RR          0.073441
CH_SVR       167.182317
CCS_GB         0.181733
CCS_MLP        0.150843
CCS_RF         0.597699
CCS_RR         0.095824
CCS_SVR       10.612309
dtype: float64


In [14]:
# SD
print(runtime_df.std())

Linear       0.011499
Nonlinear    0.026279
CH_GB        0.008558
CH_MLP       0.015450
CH_RF        0.016022
CH_RR        0.008747
CH_SVR       3.559105
CCS_GB       0.008033
CCS_MLP      0.009095
CCS_RF       0.084853
CCS_RR       0.022618
CCS_SVR      0.116743
dtype: float64


In [15]:
runtime_df.to_csv('runtime.csv',index=False)